# Hybrid Quantum PINN (HQPINN) — Flow Around a Circular Cylinder

Sandwich architecture, identical in spirit to the KdV / 2D-Burgers QAPINNs:

    (x, y) ─► classical encoder (tanh, ×π) ─► quantum circuit (data re-uploading,
              strongly-entangling) ─► classical head ─► (u, v, p)

- **Ground truth:** nutils Taylor–Hood P2/P1 FEM (`gt_out/cylinder_gt.npz`, Re = 20).
- **Baseline:** classical PINN (same 3-output architecture).
- **This notebook:** the quantum-assisted PINN + a unified comparison — loss
  curves for classical / quantum, spatial fields vs. ground truth, and a
  results table (params / accuracy / loss / timing).

**Device discipline** carried over from prior projects:
`default.qubit` + `diff_method="backprop"` (exact 2nd-order autograd), the CPU/GPU
"device dance", everything on CPU for like-for-like classical-vs-quantum timing.

> Prereq: run `ground_truth.py` first to produce `gt_out/cylinder_gt.npz`.

In [1]:
import os, time, json, numpy as np
import torch, torch.nn as nn
import pennylane as qml
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)  
torch.manual_seed(0); np.random.seed(0)

DEVICE = "cpu"  
print("pennylane", qml.version(), "| torch", torch.__version__, "| device", DEVICE)

pennylane 0.45.1 | torch 2.6.0+cu124 | device cpu


## 1. Load the FEM ground truth
Geometry, physics constants, and the dense validation point cloud all come from the nutils solver's metadata — no magic numbers duplicated here.

In [2]:
GT_PATH = "gt_out/cylinder_gt.npz"       # produced by ground_truth.py
gt = np.load(GT_PATH)

L, H   = float(gt["L"]), float(gt["H"])
XC, YC = float(gt["xc"]), float(gt["yc"])
R      = float(gt["R"])
RHO, NU, UM = float(gt["rho"]), float(gt["nu"]), float(gt["Um"])
RE     = float(gt["Re"])

GT_XY  = np.stack([gt["x"], gt["y"]], axis=1)
GT_UVP = np.stack([gt["u"], gt["v"], gt["p"]], axis=1)
print(f"Re={RE:.1f}  |  {len(gt['x'])} ground-truth points  |  "
      f"channel {L}x{H}, cylinder D={2*R} @ ({XC},{YC})")

Re=20.0  |  23756 ground-truth points  |  channel 2.2x0.41, cylinder D=0.1 @ (0.2,0.2)


## 2. Domain sampling + input normalization

Two things matter for the quantum model specifically:

1. **Normalize inputs to [-1, 1]** before the encoder — the angle embedding wants bounded features.
2. **Chain-rule scale factors** `SX=2/L, SY=2/H`: the network sees normalized coords, but the Navier–Stokes residual is in *physical* units, so every derivative is rescaled.

In [3]:
rng = np.random.default_rng(0)

def outside_cylinder(x, y, margin=1e-3):
    return (x - XC)**2 + (y - YC)**2 >= (R + margin)**2

def sample_interior(n, wake_bias=False):
    """Reject points inside the cylinder. wake_bias concentrates points in the
    near-wake / boundary layer (analogue of shock-biased collocation)."""
    pts = []
    while len(pts) < n:
        m = n - len(pts)
        if wake_bias:
            x = rng.uniform(XC - R, XC + 8*R, 2*m)
            y = rng.uniform(YC - 3*R, YC + 3*R, 2*m)
        else:
            x = rng.uniform(0, L, 2*m); y = rng.uniform(0, H, 2*m)
        k = outside_cylinder(x, y) & (x >= 0)&(x <= L)&(y >= 0)&(y <= H)
        pts += np.stack([x[k], y[k]], 1).tolist()
    return np.array(pts[:n])

def norm_xy(xy):
    return np.stack([2*xy[:,0]/L - 1, 2*xy[:,1]/H - 1], axis=1)

SX, SY = 2.0/L, 2.0/H          # d(x_norm)/dx , d(y_norm)/dy  (chain rule)
def to_t(a): return torch.tensor(a, dtype=torch.float64, device=DEVICE)

## 3. Build all training tensors
Interior collocation (bulk + wake-biased), the four boundary sets (inlet parabola, no-slip walls, no-slip cylinder, outlet pressure-gauge), and sparse FEM data supervision.

In [4]:
# collocation
Xf = to_t(norm_xy(np.concatenate([sample_interior(4000),
                                  sample_interior(1500, wake_bias=True)])))
# boundaries
n_bc = 300
y_in  = rng.uniform(0, H, n_bc)
Xin   = to_t(norm_xy(np.stack([np.zeros(n_bc), y_in], 1)))
Uin   = to_t(np.stack([4*UM*y_in*(H-y_in)/H**2, np.zeros(n_bc)], 1))   # parabola
x_w   = rng.uniform(0, L, n_bc)
Xwall = to_t(norm_xy(np.concatenate([np.stack([x_w, np.zeros(n_bc)],1),
                                     np.stack([x_w, np.full(n_bc,H)],1)])))
th    = rng.uniform(0, 2*np.pi, 500)
Xcyl  = to_t(norm_xy(np.stack([XC+R*np.cos(th), YC+R*np.sin(th)], 1)))
y_out = rng.uniform(0, H, n_bc)
Xout  = to_t(norm_xy(np.stack([np.full(n_bc,L), y_out], 1)))            # p-gauge
# sparse FEM supervision
idx = rng.choice(len(gt["x"]), 1500, replace=False)
Xd  = to_t(norm_xy(GT_XY[idx])); Ud = to_t(GT_UVP[idx])
print("collocation:", Xf.shape[0], "| data:", Xd.shape[0])

collocation: 5500 | data: 1500


## 4. Quantum node — data re-uploading + strongly-entangling ansatz

- `AngleEmbedding` (RY) tiles the `n_qubit`-dim encoder features onto all wires.
- **Data RE-UPLOADING**: features re-embedded before every layer (`l>0`). The re-upload count R is the primary expressivity knob (Schuld–Sweke–Meyer: it sets the accessible Fourier bandwidth).
- `StronglyEntanglingLayers` = trainable RX/RY/RZ + CNOT ring.
- `diff_method="backprop"` on `default.qubit` → PyTorch autograd computes exact 2nd-order derivatives (parameter-shift would be far costlier for the Laplacian).

In [5]:
def make_qnode(n_qubits, n_layers, reupload=True):
    dev = qml.device("default.qubit", wires=n_qubits)

    def _encode(x):
        idx = [i % x.shape[-1] for i in range(n_qubits)]   # tile 2 feats -> n wires
        qml.AngleEmbedding(x[..., idx], wires=range(n_qubits), rotation="Y")

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        _encode(inputs)
        for l in range(n_layers):
            if reupload and l > 0:
                _encode(inputs)
            qml.StronglyEntanglingLayers(weights[l:l+1], wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]
    return circuit

## 5. The SandwichQAPINN model

**Device dance:** `q_weights` and the statevector simulator live on CPU. `forward()` moves features to CPU before the circuit and results back **without** `detach()`/`.numpy()`, so the autograd graph survives across the CPU boundary — required for the 2nd-order PDE derivatives. Same pattern that fixed the 2D-Burgers QAPINN.

The `probs()` method exposes the raw measurement distribution for the `xai/` package (`QuantumProbe.from_qapinn()`).

In [6]:
class SandwichQAPINN(nn.Module):
    def __init__(self, n_qubits=4, n_layers=3, enc_hidden=16, head_hidden=16,
                 reupload=True):
        super().__init__()
        self.n_qubits, self.n_layers, self.reupload = n_qubits, n_layers, reupload
        self.encoder = nn.Sequential(
            nn.Linear(2, enc_hidden), nn.Tanh(),
            nn.Linear(enc_hidden, enc_hidden), nn.Tanh(),
            nn.Linear(enc_hidden, n_qubits), nn.Tanh())     # -> [-1,1]^n_qubits
        self.qnode = make_qnode(n_qubits, n_layers, reupload)
        shape = qml.StronglyEntanglingLayers.shape(n_layers, n_qubits)
        self.q_weights = nn.Parameter(0.1 * torch.randn(*shape))
        self.head = nn.Sequential(
            nn.Linear(n_qubits, head_hidden), nn.Tanh(),
            nn.Linear(head_hidden, head_hidden), nn.Tanh(),
            nn.Linear(head_hidden, 3))                       # -> (u, v, p)
        for m in list(self.encoder) + list(self.head):
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)

    def _quantum(self, feats):
        dev = feats.device
        feats = feats.to("cpu")                 # statevector sim runs on CPU
        z = self.qnode(feats, self.q_weights)
        z = torch.stack(z, dim=-1) if isinstance(z, list) else z
        return z.to(dev)                        # graph intact — no detach

    def forward(self, xy):
        feats = np.pi * self.encoder(xy)        # scale into [-pi, pi] for RY angles
        z = self._quantum(feats)                # <Z_i> in [-1,1]
        u, v, p = self.head(z).split(1, dim=1)
        return u, v, p

    def probs(self, xy_norm):
        """Measurement distribution over the 2^n basis — for the XAI interface."""
        feats = (np.pi * self.encoder(xy_norm)).to("cpu")
        dev = qml.device("default.qubit", wires=self.n_qubits)
        def _enc(x):
            idx = [i % x.shape[-1] for i in range(self.n_qubits)]
            qml.AngleEmbedding(x[..., idx], wires=range(self.n_qubits), rotation="Y")
        @qml.qnode(dev, interface="torch")
        def c(inputs, weights):
            _enc(inputs)
            for l in range(self.n_layers):
                if self.reupload and l > 0: _enc(inputs)
                qml.StronglyEntanglingLayers(weights[l:l+1], wires=range(self.n_qubits))
            return qml.probs(wires=range(self.n_qubits))
        return c(feats, self.q_weights)

### 5b. Classical PINN baseline (same 3-output interface)
A plain tanh MLP so the comparison is apples-to-apples: identical loss, identical residual, identical training loop — only the function approximator differs.

In [7]:
class ClassicalPINN(nn.Module):
    def __init__(self, width=64, depth=5):
        super().__init__()
        layers, d = [], 2
        for _ in range(depth):
            lin = nn.Linear(d, width)
            nn.init.xavier_normal_(lin.weight); nn.init.zeros_(lin.bias)
            layers += [lin, nn.Tanh()]; d = width
        out = nn.Linear(d, 3)
        nn.init.xavier_normal_(out.weight); nn.init.zeros_(out.bias)
        layers.append(out)
        self.net = nn.Sequential(*layers)
    def forward(self, xy):
        o = self.net(xy)
        return o[:,0:1], o[:,1:2], o[:,2:3]

## 6. Navier–Stokes residual (shared by both models)

Steady incompressible NS. Derivatives are taken w.r.t. the *normalized* inputs and rescaled by `SX, SY` so the residual is in physical units:

$$u u_x + v u_y + \tfrac1\rho p_x - \nu(u_{xx}+u_{yy}) = 0,\quad
  u v_x + v v_y + \tfrac1\rho p_y - \nu(v_{xx}+v_{yy}) = 0,\quad
  u_x + v_y = 0.$$

In [8]:
def ns_residual(model, Xn):
    Xn = Xn.clone().requires_grad_(True)
    u, v, p = model(Xn)
    def g(f): return torch.autograd.grad(f, Xn, torch.ones_like(f),
                                         create_graph=True)[0]
    du, dv, dp = g(u), g(v), g(p)
    u_x, u_y = du[:,0:1]*SX, du[:,1:2]*SY
    v_x, v_y = dv[:,0:1]*SX, dv[:,1:2]*SY
    p_x, p_y = dp[:,0:1]*SX, dp[:,1:2]*SY
    u_xx = g(du[:,0:1])[:,0:1]*SX*SX; u_yy = g(du[:,1:2])[:,1:2]*SY*SY
    v_xx = g(dv[:,0:1])[:,0:1]*SX*SX; v_yy = g(dv[:,1:2])[:,1:2]*SY*SY
    r_u = u*u_x + v*u_y + p_x/RHO - NU*(u_xx + u_yy)   # momentum-x
    r_v = u*v_x + v*v_y + p_y/RHO - NU*(v_xx + v_yy)   # momentum-y
    r_c = u_x + v_y                                    # continuity
    return r_u, r_v, r_c

## 7. Unified training loop

One function trains **either** model with the same composite loss
`L = w_pde·L_pde + w_bc·L_bc + w_gauge·L_gauge + w_data·L_data`, two-phase
(Adam warm-up → L-BFGS refine), and records a history keyed to a global
iteration counter so the curves overlay cleanly.

> **Ablation note (mandatory per the honesty guardrails):** for a real
> quantum-contribution claim you'd also run the *classical twin* (encoder+head,
> quantum layer replaced by identity) and the *frozen-circuit* arm
> (`q_weights.requires_grad_(False)`). Both slot straight into this loop.

In [9]:
def train_model(model, tag, adam_iters=1500, adam_lr=3e-3, lbfgs_iters=300,
                w_pde=1.0, w_bc=10.0, w_data=5.0, w_gauge=1.0, log_every=100):
    mse = nn.MSELoss()
    hist = {"it": [], "loss": [], "pde": [], "bc": [], "data": []}

    def losses():
        r_u, r_v, r_c = ns_residual(model, Xf)
        L_pde = (mse(r_u, torch.zeros_like(r_u)) + mse(r_v, torch.zeros_like(r_v))
                 + mse(r_c, torch.zeros_like(r_c)))
        ui, vi, _  = model(Xin);  uw, vw, _ = model(Xwall)
        uc, vc, _  = model(Xcyl); _, _, po  = model(Xout)
        z = torch.zeros_like
        L_bc = (mse(torch.cat([ui,vi],1), Uin)
                + mse(torch.cat([uw,vw],1), z(torch.cat([uw,vw],1)))
                + mse(torch.cat([uc,vc],1), z(torch.cat([uc,vc],1))))
        L_gauge = mse(po, torch.zeros_like(po))
        ud, vd, pd = model(Xd); L_data = mse(torch.cat([ud,vd,pd],1), Ud)
        return L_pde, L_bc, L_gauge, L_data

    def total():
        Lp, Lb, Lg, Ld = losses()
        return w_pde*Lp + w_bc*Lb + w_gauge*Lg + w_data*Ld, (Lp, Lb, Lg, Ld)

    t0 = time.time(); git = 0
    opt = torch.optim.Adam(model.parameters(), lr=adam_lr)
    for it in range(adam_iters):
        opt.zero_grad(); loss, (Lp,Lb,Lg,Ld) = total()
        loss.backward(); opt.step(); git += 1
        if it % log_every == 0 or it == adam_iters-1:
            hist["it"].append(git); hist["loss"].append(loss.item())
            hist["pde"].append(Lp.item()); hist["bc"].append(Lb.item())
            hist["data"].append(Ld.item())
            print(f"[{tag} adam {it:4d}] loss={loss.item():.3e} "
                  f"pde={Lp.item():.2e} bc={Lb.item():.2e} data={Ld.item():.2e}")

    if lbfgs_iters > 0:
        opt = torch.optim.LBFGS(model.parameters(), max_iter=lbfgs_iters,
                                history_size=50, line_search_fn="strong_wolfe",
                                tolerance_grad=1e-12, tolerance_change=1e-14)
        def closure():
            opt.zero_grad(); loss, _ = total(); loss.backward(); return loss
        opt.step(closure); git += lbfgs_iters

    final, (Lp,Lb,Lg,Ld) = total()
    hist["it"].append(git); hist["loss"].append(final.item())
    hist["pde"].append(Lp.item()); hist["bc"].append(Lb.item()); hist["data"].append(Ld.item())
    train_time = time.time() - t0

    with torch.no_grad():
        Xall = to_t(norm_xy(GT_XY))
        up, vp, pp = model(Xall)
        pred = torch.cat([up, vp], 1).cpu().numpy(); true = GT_UVP[:, :2]
        rel_l2 = float(np.linalg.norm(pred - true) / np.linalg.norm(true))
    n_params = sum(p.numel() for p in model.parameters())
    print(f"[{tag}] final loss={final.item():.3e}  rel-L2={rel_l2:.4e}  "
          f"params={n_params}  time={train_time:.1f}s")
    return dict(tag=tag, hist=hist, rel_l2=rel_l2, final_loss=final.item(),
                n_params=n_params, train_time=train_time, model=model)

## 8. Train the classical baseline
Full schedule (Adam → L-BFGS). Reduce `adam_iters` for a quick smoke test.

In [11]:
res_classical = train_model(ClassicalPINN(width=64, depth=5),
                            "Classical PINN",
                            adam_iters=2000, lbfgs_iters=0)

[Classical PINN adam    0] loss=1.037e+00 pde=9.14e-02 bc=8.36e-02 data=2.14e-02
[Classical PINN adam  100] loss=2.060e-01 pde=3.36e-04 bc=1.55e-02 data=1.02e-02
[Classical PINN adam  200] loss=2.055e-01 pde=3.03e-04 bc=1.54e-02 data=1.03e-02
[Classical PINN adam  300] loss=2.049e-01 pde=2.89e-04 bc=1.53e-02 data=1.03e-02
[Classical PINN adam  400] loss=2.040e-01 pde=2.91e-04 bc=1.52e-02 data=1.03e-02
[Classical PINN adam  500] loss=2.022e-01 pde=3.18e-04 bc=1.50e-02 data=1.03e-02
[Classical PINN adam  600] loss=1.966e-01 pde=4.71e-04 bc=1.45e-02 data=1.02e-02
[Classical PINN adam  700] loss=1.686e-01 pde=9.10e-03 bc=1.08e-02 data=1.02e-02
[Classical PINN adam  800] loss=1.972e-01 pde=1.47e-03 bc=1.51e-02 data=8.91e-03
[Classical PINN adam  900] loss=1.613e-01 pde=3.50e-03 bc=1.21e-02 data=7.34e-03
[Classical PINN adam 1000] loss=1.471e-01 pde=5.71e-03 bc=1.08e-02 data=6.62e-03
[Classical PINN adam 1100] loss=1.352e-01 pde=6.67e-03 bc=1.01e-02 data=5.58e-03
[Classical PINN adam 1200] l

## 9. Train the QAPINN
4 qubits, 3 strongly-entangling layers, data re-uploading on. The circuit sim dominates wall-clock, so keep iteration counts modest (the same stable-but-capacity-limited behaviour seen on Burgers is expected).

In [ ]:
res_quantum = train_model(SandwichQAPINN(n_qubits=4, n_layers=2, reupload=True),
                          "QAPINN 4q",
                          adam_iters=2000, adam_lr=5e-3, lbfgs_iters=0)

[QAPINN 4q adam    0] loss=3.341e+01 pde=2.92e+01 bc=3.60e-01 data=1.19e-01
[QAPINN 4q adam  100] loss=1.775e-01 pde=1.05e-02 bc=1.14e-02 data=1.06e-02
[QAPINN 4q adam  200] loss=1.534e-01 pde=1.62e-02 bc=8.39e-03 data=1.07e-02
[QAPINN 4q adam  300] loss=1.396e-01 pde=1.95e-02 bc=6.76e-03 data=1.05e-02
[QAPINN 4q adam  400] loss=1.285e-01 pde=2.11e-02 bc=5.86e-03 data=9.74e-03
[QAPINN 4q adam  500] loss=1.168e-01 pde=2.30e-02 bc=5.33e-03 data=8.09e-03
[QAPINN 4q adam  600] loss=1.049e-01 pde=2.67e-02 bc=4.93e-03 data=5.79e-03
[QAPINN 4q adam  700] loss=1.003e-01 pde=2.90e-02 bc=4.64e-03 data=4.97e-03
[QAPINN 4q adam  800] loss=9.764e-02 pde=2.87e-02 bc=4.45e-03 data=4.89e-03
[QAPINN 4q adam  900] loss=9.778e-02 pde=2.93e-02 bc=4.19e-03 data=5.32e-03
[QAPINN 4q adam 1000] loss=9.350e-02 pde=2.80e-02 bc=4.15e-03 data=4.79e-03
[QAPINN 4q adam 1100] loss=9.103e-02 pde=2.77e-02 bc=3.99e-03 data=4.68e-03
[QAPINN 4q adam 1200] loss=8.844e-02 pde=2.72e-02 bc=3.89e-03 data=4.46e-03
[QAPINN 4q a

## 10. Overlaid training-loss curves — classical vs quantum
Left: total loss. Right: PDE (solid) and data (dashed) components, so you can see *which* term each model is limited by.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for res, c in [(res_classical, "tab:blue"), (res_quantum, "tab:red")]:
    h = res["hist"]
    ax[0].semilogy(h["it"], h["loss"], '-o', color=c, ms=3, label=res["tag"])
ax[0].set_xlabel("iteration"); ax[0].set_ylabel("total loss")
ax[0].set_title("Total training loss"); ax[0].legend(); ax[0].grid(alpha=.3)

for res, c in [(res_classical, "tab:blue"), (res_quantum, "tab:red")]:
    h = res["hist"]
    ax[1].semilogy(h["it"], h["pde"], '-', color=c, label=f"{res['tag']} PDE")
    ax[1].semilogy(h["it"], h["data"], '--', color=c, alpha=.6, label=f"{res['tag']} data")
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("loss component")
ax[1].set_title("Loss components (PDE solid, data dashed)")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.savefig("qapinn_out/loss_compare.png", dpi=130) if os.path.isdir("qapinn_out") else None
plt.show()

## 11. Spatial fields — Ground truth vs Classical vs QAPINN
Velocity magnitude on the FEM point cloud, all three on a shared colour scale so under-prediction in the wake is directly visible.

In [ ]:
def umag_pred(model):
    with torch.no_grad():
        up, vp, pp = model(to_t(norm_xy(GT_XY)))
    return np.linalg.norm(np.concatenate([up.cpu().numpy(), vp.cpu().numpy()], 1), axis=1)

umag_true = np.linalg.norm(GT_UVP[:, :2], axis=1)
panels = [(umag_true,                       "|u| Ground Truth (FEM)"),
          (umag_pred(res_classical["model"]),"|u| Classical PINN"),
          (umag_pred(res_quantum["model"]),  "|u| QAPINN 4q")]

fig, ax = plt.subplots(3, 1, figsize=(11, 7.5), constrained_layout=True)
vmax = umag_true.max()
for a, (f, t) in zip(ax, panels):
    sc = a.scatter(gt["x"], gt["y"], c=f, s=2, cmap="viridis", vmin=0, vmax=vmax)
    a.set_aspect("equal"); a.set_title(t); a.set_xlim(0, L); a.set_ylim(0, H)
    fig.colorbar(sc, ax=a, fraction=0.02)
plt.show()

## 12. Compiled results table
Parameters, final loss, accuracy (relative L2 on velocity vs. FEM truth), and wall-clock training time.

In [ ]:
import pandas as pd

def row(res):
    return {"Model": res["tag"],
            "Params": res["n_params"],
            "Final loss": f"{res['final_loss']:.3e}",
            "rel-L2 (u,v)": f"{res['rel_l2']:.3e}",
            "Train time (s)": f"{res['train_time']:.1f}",
            "s / 1k iters": f"{1000*res['train_time']/max(res['hist']['it'][-1],1):.1f}"}

df = pd.DataFrame([row(res_classical), row(res_quantum)])
df.insert(1, "Re", int(RE))
display(df)
df.to_csv("qapinn_out/results_table.csv", index=False) if os.path.isdir("qapinn_out") else None
print(df.to_string(index=False))

## 13. (Optional) Matplotlib table for figures/exports

In [ ]:
cols = ["Model", "Params", "Final loss", "rel-L2 (u,v)", "Time (s)"]
data = [[res_classical["tag"], res_classical["n_params"],
         f"{res_classical['final_loss']:.3e}", f"{res_classical['rel_l2']:.3e}",
         f"{res_classical['train_time']:.1f}"],
        [res_quantum["tag"], res_quantum["n_params"],
         f"{res_quantum['final_loss']:.3e}", f"{res_quantum['rel_l2']:.3e}",
         f"{res_quantum['train_time']:.1f}"]]
fig, ax = plt.subplots(figsize=(9, 1.6)); ax.axis("off")
t = ax.table(cellText=data, colLabels=cols, loc="center", cellLoc="center")
t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1, 1.6)
for j in range(len(cols)):
    t[0, j].set_facecolor("#40466e"); t[0, j].set_text_props(color="w", weight="bold")
plt.show()

## 14. Interpretation & next steps

Expected pattern (consistent with the 1D/2D Burgers findings):

- **Classical PINN** reaches low error (10⁻³–10⁻⁴ with the full schedule); the QAPINN trains *stably* (healthy gradients, no barren plateau) but **saturates at a higher plateau** — a **capacity/measurement bottleneck**, not an optimization failure.
- The QAPINN uses **far fewer parameters** (≈10² vs ≈10⁴) but the small register limits expressivity: the predicted field is smoother, under-resolving the near-wake gradients.

**Levers to test next** (each is one flag in this notebook):
1. **Re-upload count R** — the primary Fourier-bandwidth knob (Schuld–Sweke–Meyer).
2. **Qubit count** — 4 → 5 → 6 (watch the `s/1k iters` column climb).
3. **Ablation arms** — classical twin + frozen circuit, to attribute any gain.

**XAI hand-off:** `QuantumProbe.from_qapinn(res_quantum["model"])` plugs this model straight into the `xai/` package for the Layer-2/Layer-3 analyses (Meyer–Wallach Q, von Neumann entropy, Fourier bounds, Hessian/barren-plateau).